# 02 - Feature engineering (next-week forecast)
**Target:** next week's `cases` in the same district (`target_next_cases`), trained as `log1p(...)` (`target_log`).
**Dropped (leakage / redundant):** `is_epidemic_year`, `incidence_rate`, `cases_zscore`; the file's own `cases_lag1`, `rainfall_lag2`, `rainfall_lag4` and `year`/`week` are replaced by features built from the date.
The code lives in `src/features.py`; this notebook runs it and checks it.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import sys, numpy as np, pandas as pd
sys.path.append("../src")
from features import load_raw, build_features, DISTRICT, DATE, CASES
raw = load_raw("../data/raw/dengue_srilanka_full_2006_2025.csv")
feat, groups = build_features(raw)
features = [c for cols in groups.values() for c in cols]
print("raw rows:", len(raw), "-> feature rows:", len(feat), "| dropped:", len(raw) - len(feat))
print("missing values left in features/target:", int(feat[features + ["target_log"]].isna().sum().sum()))

raw rows: 24775 -> feature rows: 24550 | dropped: 225
missing values left in features/target: 0


## Sanity check: the series is evenly spaced, so a 'lag of 1 row' really is 'one week back'

In [2]:
# AI-GENERATED (Claude via Antigravity)
step = raw.groupby(DISTRICT)[DATE].diff().dt.days.dropna()
print(step.value_counts().to_dict(), "(7 = normal; 6 and 8 are the date jitter seen in the audit)")
assert step.between(6, 8).all()

{7.0: 24700, 8.0: 25, 6.0: 25} (7 = normal; 6 and 8 are the date jitter seen in the audit)


## Tests (asserts)
1. For 3 random rows, `cases_lag_1` equals the previous week's cases **of the same district**, and the target equals next week's cases.
2. **Leak test:** for the same rows we inflate that week's cases by +1000 and rebuild everything. If any feature used the current week's cases, it would change. None may change, while last week's *target* must change (proves the test can detect a change).

In [3]:
# AI-GENERATED (Claude via Antigravity)
rng = np.random.default_rng(42)
picks = feat.iloc[rng.choice(len(feat), 3, replace=False)]
key = lambda f, d, t: f[(f[DISTRICT] == d) & (f[DATE] == t)].iloc[0]   # find one row by district + date
for _, r in picks.iterrows():
    i = raw.index[(raw[DATE] == r[DATE]) & (raw[DISTRICT] == r[DISTRICT])][0]
    prev, nxt = raw.loc[i - 1], raw.loc[i + 1]
    assert prev[DISTRICT] == nxt[DISTRICT] == r[DISTRICT]                       # neighbours are the same district
    assert r["cases_lag_1"] == prev[CASES], "lag_1 is not last week's cases"
    assert r["target_next_cases"] == nxt[CASES], "target is not next week's cases"

    # Leak test: inflate this week's cases by 1000, rebuild all features, compare this row
    changed = raw.copy(); changed.loc[i, CASES] += 1000
    f2, _ = build_features(changed)
    old, new = key(feat, r[DISTRICT], r[DATE]), key(f2, r[DISTRICT], r[DATE])
    for c in features:
        assert old[c] == new[c], f"feature {c} depends on the current week's cases!"
    # Last week's TARGET is this week's cases, so it MUST change (proves the test can detect changes)
    assert key(f2, r[DISTRICT], prev[DATE])["target_next_cases"] == key(feat, r[DISTRICT], prev[DATE])["target_next_cases"] + 1000
    print("OK:", r[DISTRICT], r[DATE].date())
print("All tests passed")

OK: Matara 2013-12-21


OK: Badulla 2011-06-18


OK: Nuwara Eliya 2013-08-31
All tests passed


## Save and list the features by type

In [4]:
# AI-GENERATED (Claude via Antigravity)
feat.to_csv("../data/processed/features.csv", index=False)
print("Saved data/processed/features.csv", feat.shape)
for name, cols in groups.items():
    print(f"\n{name} ({len(cols)}): {cols}")
print("\nNot model inputs (kept for splitting / scoring):", [DATE, CASES, "target_next_cases", "target_log"])
print("Total features:", len(features))

Saved data/processed/features.csv (24550, 50)

case lags (5): ['cases_lag_1', 'cases_lag_2', 'cases_lag_3', 'cases_lag_4', 'cases_lag_8']

rolling past cases (4): ['cases_roll_mean_4', 'cases_roll_max_4', 'cases_roll_mean_8', 'cases_roll_max_8']

lagged climate (32): ['rainfall_mm_lag_1', 'rainfall_mm_lag_2', 'rainfall_mm_lag_3', 'rainfall_mm_lag_4', 'rainfall_mm_lag_5', 'rainfall_mm_lag_6', 'rainfall_mm_lag_7', 'rainfall_mm_lag_8', 'temp_max_c_lag_1', 'temp_max_c_lag_2', 'temp_max_c_lag_3', 'temp_max_c_lag_4', 'temp_max_c_lag_5', 'temp_max_c_lag_6', 'temp_max_c_lag_7', 'temp_max_c_lag_8', 'temp_min_c_lag_1', 'temp_min_c_lag_2', 'temp_min_c_lag_3', 'temp_min_c_lag_4', 'temp_min_c_lag_5', 'temp_min_c_lag_6', 'temp_min_c_lag_7', 'temp_min_c_lag_8', 'humidity_pct_lag_1', 'humidity_pct_lag_2', 'humidity_pct_lag_3', 'humidity_pct_lag_4', 'humidity_pct_lag_5', 'humidity_pct_lag_6', 'humidity_pct_lag_7', 'humidity_pct_lag_8']

seasonality (2): ['week_sin', 'week_cos']

district and size (3): 

## Reference: the persistence baseline (next week = this week) on all rows

In [5]:
# AI-GENERATED (Claude via Antigravity)
err = (feat["target_next_cases"] - feat[CASES]).abs()
print("Persistence MAE (all rows, original scale): %.2f | RMSE: %.2f" % (err.mean(), np.sqrt(((feat['target_next_cases'] - feat[CASES])**2).mean())))

Persistence MAE (all rows, original scale): 10.92 | RMSE: 28.06
